# **University Admission Prediction + QS-Aware Top-K Recommender**

## Objective

	•	Predicts admission outcome (0/1) with academic features + context
	•	Enriches predictions with prestige signals (QS_Universities_Ranking) 
	•	Map degree → mapped_major (1-to-N) and duplicate rows when multiple categories apply
	•	Generate Top-K recommendations using the model’s admission probability and QS prestige

# Data loading and cleaning

### Imports

In [1]:
import sys
import joblib
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, classification_report
from sklearn.model_selection import KFold, cross_validate, GroupShuffleSplit, GroupKFold, cross_val_predict, train_test_split
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
import matplotlib.pyplot as plt

### Environment

In [ ]:


NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR

for _ in range(6):
    if (PROJECT_ROOT / "src").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_EXISTS:", (PROJECT_ROOT / "src").exists())
print("SRC_INIT_EXISTS:", (PROJECT_ROOT / "src" / "__init__.py").exists())

sys.path.insert(0, str(PROJECT_ROOT))
print("sys.path[0]:", sys.path[0])

NOTEBOOK_DIR: /Users/danimorera/Documents/Portfolio/FLRecomendator/notebooks
PROJECT_ROOT: /Users/danimorera/Documents/Portfolio/FLRecomendator
SRC_EXISTS: True
SRC_INIT_EXISTS: True
sys.path[0]: /Users/danimorera/Documents/Portfolio/FLRecomendator


### Imports and Global Configuration

In [3]:
from src.config import RAW_DIR, RANDOM_STATE
from src.utils import load_excel_safe, load_qs_excel

RAW_DIR, RANDOM_STATE

(PosixPath('/Users/danimorera/Documents/Portfolio/FLRecomendator/data/raw'),
 42)

### Data Sources

In [4]:
STUDENTS_PATH = RAW_DIR / "Five Lands Stats.xlsx"
QS_PATH = RAW_DIR / "University Ranking by Major 2025.xlsx"

STUDENTS_PATH.exists(), QS_PATH.exists()

df_students_raw = load_excel_safe(STUDENTS_PATH)
df_qs_raw = load_qs_excel(QS_PATH)

In [5]:
df_students_raw.columns

Index(['TYPE OF STUDENT', 'LAST YEAR INSTITUTION', 'SCHOOL\nCURRICULUM',
       'SCHOOL\nGRADE', 'GPA', 'TOEFL \n(MAX)', 'SAT\n(SUPERSCORE)',
       'PROFILE\n(4-16)', 'PEAK', 'Unnamed: 9', 'UNIVERSITY', 'COUNTRY',
       'DEGREE', 'AREA OF STUDY', 'Unnamed: 14', 'ADMISSION', 'CURRENCY',
       'TUITION\nYEAR', 'AID\nYEAR', 'AID\nYEAR (EUR)', 'AID\nYEAR (USD)',
       'FINAL TUITION\nYEAR', 'FINAL TUITION\nYEAR (EUR)',
       'FINAL TUITION\nYEAR (USD)', '% AID', 'Unnamed: 25', 'Unnamed: 26',
       'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29'],
      dtype='object')

### Clean student data 

In [6]:
from src.cleaning_students import clean_students_schema, null_students_schema

df_students = clean_students_schema(df_students_raw)
df_students = null_students_schema(df_students)


### Clean QS data

In [7]:
from src.cleaning_qs import initial_clean_qs_schema, augmentation_qs_schema, clean_qs_schema

df_qs = clean_qs_schema(augmentation_qs_schema(initial_clean_qs_schema(df_qs_raw)))

# Degree mapping & merge

### Mapping of df_qs, df_students (degree / universities)

In [8]:
from src.merge_students_qs import university_mapping, degree_mapping, show_degree_mapping

df_students["university"] = [university_mapping(university) for university in df_students["university"]]
df_qs["university"] = [university_mapping(university) for university in df_qs["university"]]

df_students, df_qs_degrees = degree_mapping(df_students)

### Mapped degrees

In [9]:
show_degree_mapping(df_qs_degrees)


Computer Science (15):
  - Computer Science
  - Electronic Engineering with Computer Systems H6G4
  - Business, Cognitive Science, Computer Animation, Computer Science & Web Design, Data Science & Data Analytics, Economics, Engineering, Management
  - Computer Science & Mathematics
  - Mathematics & Computer Science
  - Computer Science and Mathematics with Industrial Experience
  - Computer Engineering
  - Comptuter Science and Artificial Intellience
  - Electrical & Computer Engineering
  - Computer Science and Business
  - Business Information Technology
  - Computing Science & Mathematics
  - Mathematics and Computer Science 3FT BEng (Hon)
  - Mathematics - Computer Science, B.S.
  - Computer Science & Web Design, Engineering, 

Data Science (9):
  - Data & Business Analytics
  - Biosystems Analytics & Technology
  - Ingeniería de Telecomunicaciones  y Business Analytics
  - Business Analytics
  - ADE+ Business Analytics
  - Doble Grado ADE y Ciencias de Datos
  - BBA & Data Analy

### Universidades en comun df_students/df_qs

In [10]:
from src.merge_students_qs import show_inner_universities

show_inner_universities(df_students, df_qs)

Universidades únicas students: 214
Universidades únicas qs: 1758
Universidades en común: 113


There's only 113 unique universities in the student dataset that appear in the QS Rankings, meaning we might not be able to give a QS score to the remaining 101 universities. This means we get NaN qs_score rows that will have to be deleted from the training dataset, as we do not have any raw information on their prestige.

In [ ]:
from src.merge_students_qs import merge_students_qs

df_pred = merge_students_qs(df_students, df_qs)
df_pred.shape

In [14]:
df_pred.isnull().sum()

school_curriculum      0
school_grade           0
gpa                    0
toefl                  0
sat                    0
profile_score          0
peak                   0
university             0
country                0
degree                 0
admitted               0
sat_required           0
qs_score             369
dtype: int64

In [16]:
df_train = df_pred[df_pred["qs_score"].notna()].copy()

# Feature Engineering

In [ ]:
from src.features import add_student_id

df_pred = add_student_id(df_pred)
df_pred["student_id"].nunique(), df_pred.shape

(110, (751, 14))

In [17]:
target = "admitted"
X = df_train[df_train["admitted"].notnull()].drop(columns=["student_id", target, "university"])
y = df_train[df_train["admitted"].notnull()][target]

In [18]:
FEATURE_COLS = X.columns.tolist()

In [19]:
groups = df_train["student_id"].astype(str)

gSplit = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gSplit.split(X, y, groups=groups))


In [20]:
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test  = groups.iloc[test_idx]

# Model Training

In [21]:
from src.modeling import preprocessor_schema, baseline_model

preprocess = preprocessor_schema(X_train)
baseline_pipe = baseline_model(preprocess, random_state=42)

baseline_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

### Evaluation

In [22]:
from src.evaluation import evaluate_model

metrics = evaluate_model(baseline_pipe, X_test, y_test)
metrics["roc_auc"], metrics["pr_auc"], metrics["f1"]
print(metrics["report"])

              precision    recall  f1-score   support

         0.0      0.714     0.152     0.250        33
         1.0      0.741     0.976     0.842        82

    accuracy                          0.739       115
   macro avg      0.728     0.564     0.546       115
weighted avg      0.733     0.739     0.672       115



In [23]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
import numpy as np

proba = baseline_pipe.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC:", average_precision_score(y_test, proba))
print("mean proba:", proba.mean(), "min/max:", proba.min(), proba.max())

ROC-AUC: 0.7695861049519586
PR-AUC: 0.8811512508720691
mean proba: 0.729937548011937 min/max: 0.4139609544091009 0.9653997249847747


## Param Tuning 

Evolution from baseline_model, to best_model. Same pipeline with a RF with optimized parameters.

In [26]:
from src.modeling import preprocessor_schema, baseline_model, rf_tuning

search = rf_tuning(baseline_pipe, X_train, y_train, groups_train, n_iter=50, scoring="average_precision")
best_model = search.best_estimator_

search.best_score_, search.best_params_

Fitting 5 folds for each of 50 candidates, totalling 250 fits


(np.float64(0.9170107389070253),
 {'model__n_estimators': 1000,
  'model__min_samples_split': 2,
  'model__min_samples_leaf': 10,
  'model__max_features': 'log2',
  'model__max_depth': 5,
  'model__class_weight': None,
  'model__bootstrap': True})

### Cross-Validation

In [27]:
import numpy as np
from sklearn.model_selection import GroupKFold, cross_val_predict

cv = GroupKFold(n_splits=5)

proba_oof = cross_val_predict(
    best_model,
    X_train, y_train,
    cv=cv,
    groups=groups_train,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

proba_oof.min(), proba_oof.mean(), proba_oof.max()

(np.float64(0.559960161243559),
 np.float64(0.7837442324759619),
 np.float64(0.9470026443582515))

# Threshold tuning

Threshold tuning test for reducing false positives as we have an unbalanced model which is predicting to many 1s. We use this so that the topk model doesn't recomend any universities with proba < threshold, and then aplies any order for final recomendation

In [28]:
from sklearn.metrics import precision_score, recall_score

thresholds = np.linspace(0.05, 0.95, 91)
min_precision = 0.85

best_t = None
best_rec = -1
best_prec = None

for t in thresholds:
    pred = (proba_oof >= t).astype(int)
    prec = precision_score(y_train, pred, zero_division=0)
    rec = recall_score(y_train, pred, zero_division=0)
    if prec >= min_precision and rec > best_rec:
        best_rec = rec
        best_prec = prec
        best_t = t

best_t, best_prec, best_rec

(np.float64(0.82), 0.8725490196078431, 0.4238095238095238)

In [29]:
pred_oof = (proba_oof >= best_t).astype(int)
print("OOF threshold:", best_t)
print(classification_report(y_train, pred_oof, digits=3))
print(confusion_matrix(y_train, pred_oof))

OOF threshold: 0.82
              precision    recall  f1-score   support

         0.0      0.267     0.772     0.396        57
         1.0      0.873     0.424     0.571       210

    accuracy                          0.498       267
   macro avg      0.570     0.598     0.483       267
weighted avg      0.743     0.498     0.533       267

[[ 44  13]
 [121  89]]


### Final Evaluation

In [30]:
from sklearn.metrics import roc_auc_score, average_precision_score

best_model.fit(X_train, y_train)

proba_test = best_model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= best_t).astype(int)

print("Test ROC-AUC:", roc_auc_score(y_test, proba_test))
print("Test PR-AUC:", average_precision_score(y_test, proba_test))
print("Final threshold:", best_t)
print(classification_report(y_test, pred_test, digits=3))
print(confusion_matrix(y_test, pred_test))

Test ROC-AUC: 0.7636733185513673
Test PR-AUC: 0.8815639614969861
Final threshold: 0.82
              precision    recall  f1-score   support

         0.0      0.474     0.818     0.600        33
         1.0      0.897     0.634     0.743        82

    accuracy                          0.687       115
   macro avg      0.685     0.726     0.671       115
weighted avg      0.775     0.687     0.702       115

[[27  6]
 [30 52]]


# Model export

In [31]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

bundle = {
    "model": best_model,
    "threshold": 0.82,
    "feature_cols": FEATURE_COLS,
}

joblib.dump(bundle, MODELS_DIR / "admission_prob_model.joblib")

['/Users/danimorera/Documents/Portfolio/FLRecomendator/models/admission_prob_model.joblib']

### Export QS catalog

In [33]:
df_qs_catalog = (
    df_qs[["university","country","degree","qs_score"]]
    .dropna(subset=["qs_score"])
    .drop_duplicates()
    .reset_index(drop=True)
)

(df_qs_catalog).to_csv(PROJECT_ROOT / "reports" / "tables" / "qs_catalog.csv", index=False)